# Project 3: A/B Testing & Bayesian Experimentation

This notebook covers the standard workflow for product experimentation: Pre-experiment Power Analysis, Frequentist Testing, and Bayesian Posterior analysis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import beta, norm
import sys
sys.path.append('../src')
from stats_utils import perform_z_test, calculate_mde

plt.style.use('fivethirtyeight')
print("A/B Testing Simulator Ready")

## 1. Pre-Experiment: Sample Size & Power Analysis

Before running an experiment, we must determine the required sample size to detect a Minimum Detectable Effect (MDE).

In [ ]:
sample_sizes = np.linspace(100, 10000, 50)
mdes = [calculate_mde(baseline_conversion=0.1, sample_size=int(n)) for n in sample_sizes]

plt.figure(figsize=(12, 6))
plt.plot(sample_sizes, mdes, lw=2)
plt.axhline(0.015, color='red', linestyle='--', label='Target MDE (1.5%)')
plt.title('Sample Size vs Minimum Detectable Effect')
plt.xlabel('Sample Size per Variant')
plt.ylabel('MDE (Absolute)')
plt.legend()
plt.grid(True)
plt.show()

## 2. Bayesian Posterior Distributions

Visualizing the 'uncertainty' in our conversion estimates for Control (A) and Treatment (B).

In [ ]:
conv_a, size_a = 520, 5000
conv_b, size_b = 580, 5000

x = np.linspace(0.08, 0.14, 1000)
y_a = beta.pdf(x, conv_a + 1, size_a - conv_a + 1)
y_b = beta.pdf(x, conv_b + 1, size_b - conv_b + 1)

plt.figure(figsize=(12, 6))
plt.plot(x, y_a, label='Control (A)', color='blue')
plt.plot(x, y_b, label='Treatment (B)', color='green')
plt.fill_between(x, 0, y_a, alpha=0.1, color='blue')
plt.fill_between(x, 0, y_b, alpha=0.1, color='green')
plt.title('Beta Posterior Distributions')
plt.xlabel('Conversion Rate')
plt.legend()
plt.show()

## 3. Probability Density of Lift

Instead of a single point estimate, we look at the distribution of the relative lift.

In [ ]:
a_samples = np.random.beta(conv_a + 1, size_a - conv_a + 1, 10000)
b_samples = np.random.beta(conv_b + 1, size_b - conv_b + 1, 10000)
lift_samples = (b_samples - a_samples) / a_samples

plt.figure(figsize=(10, 6))
sns.kdeplot(lift_samples, fill=True, color='purple')
plt.axvline(0, color='red', linestyle='--')
plt.title('Distribution of Relative Lift (B vs A)')
plt.xlabel('Relative Lift')
plt.show()

print(f"Probability B is better than A: {(lift_samples > 0).mean():.2%}")
print(f"Expected relative lift: {lift_samples.mean():.2%}")